# Ablation V6 — Paper-grade (sobre PINNModal_v4 original)

Estudio de ablación acumulativo sobre la arquitectura **original** (PINNModal_v4b),
desactivando un componente por fila. Régimen de entrenamiento **idéntico al main**
(`Adam + CosineAnnealingWarmRestarts`, 600 ép., checkpoint best-val ≥250).

### Escalera de 6 filas (cada una añade exactamente un elemento)
| # | Variante | Único cambio vs fila anterior |
|---|---|---|
| 1 | Supervised NN | — (piso de referencia, sin encoder T₁, solo datos) |
| 2 | + Encoder T₁ dedicado | arquitectura v4b completa (aún solo datos) |
| 3 | + L_eig | residuo de autovalores Kφ=ω²Mφ |
| 4 | + L_φ (MAC) | similitud de formas modales |
| 5 | + L_ortho | ortonormalidad M-ponderada → loss completa, curriculum plano |
| 6 | Full PINN v4 | curriculum staged v20 = **modelo del paper** |

### Garantías metodológicas
- **Mismos índices que el loop principal**: `default_rng(SEED + 16·100)`, Trial 16.
- **Anti-fuga verificada por código**: assert de intersección vacía train/val/test por rep.
- **Scalers solo desde train**: `compute_scalers(idx_tr)` antes de construir val/test.
- **Candado de reproducción**: la fila 6 debe caer dentro de ±1σ de la Tabla 8 del Trial 16.
  Si pasa → el resto de la tabla es interpretable. Si no → hay discrepancia de pipeline.
- **λ por término = los del main** (eig=0.1, phi=0.1, ortho=1e-5, ratio=1.0). Cero tuning por fila.

> **Nota de honestidad:** se eliminó el parámetro muerto `T1` (estaba en los dicts pero
> `compute_loss_v16` nunca lo usa en la suma de pérdidas). No había confound que corregir.

Costo: 6 filas × 4 reps × 600 ép. en Trial 16 (~1.5 h en RTX 3070 Ti).
Pon `RUN_ABL_V6 = True` para ejecutar.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (A) Arquitectura SIN encoder_T1 dedicado (filas 1-2)
# Misma capacidad/hidden que v4b; T1 sale del encoder compartido.
# ══════════════════════════════════════════════════════════════════════
class PINNModal_noT1enc(nn.Module):
    """Variante de PINNModal_v4 donde T1 sale del encoder compartido."""
    def __init__(self, n_inputs, n_modos=18, n_max_pisos=18, hidden=256):
        super().__init__()
        self.n_modos     = n_modos
        self.n_max_pisos = n_max_pisos
        self.encoder = nn.Sequential(
            nn.Linear(n_inputs, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            ResBlock(hidden), ResBlock(hidden),
            ResBlock(hidden), ResBlock(hidden),
        )
        self.head_T1 = nn.Sequential(
            nn.Linear(hidden, 32), nn.SiLU(), nn.Linear(32, 1))
        self.head_T_rest = nn.Sequential(
            nn.Linear(hidden, 128), nn.SiLU(), nn.Linear(128, n_modos - 1))
        self.head_Phi = nn.Sequential(
            nn.Linear(hidden, 512), nn.SiLU(),
            nn.Linear(512, n_modos * n_max_pisos * 3))
        self.head_resp = nn.Sequential(
            nn.Linear(hidden, 256), nn.SiLU(),
            nn.Linear(256, n_max_pisos * 4))
        self.head_Vb = nn.Sequential(
            nn.Linear(hidden, 64), nn.SiLU(), nn.Linear(64, 2))

    def forward(self, X, mask):
        B  = X.shape[0]
        h  = self.encoder(X)
        T1 = self.head_T1(h)
        T_deltas = F.softplus(self.head_T_rest(h))
        parts = [T1]
        for r in range(self.n_modos - 1):
            parts.append(parts[-1] - T_deltas[:, r:r+1])
        logT_pred = torch.cat(parts, dim=1)
        Phi_raw  = self.head_Phi(h).view(B, self.n_modos, self.n_max_pisos, 3)
        mask_4d  = mask.unsqueeze(1).unsqueeze(-1)
        Phi_raw  = Phi_raw * mask_4d
        Phi_norm = Phi_raw / Phi_raw.norm(dim=2, keepdim=True).clamp(min=1e-8)
        Phi_all  = Phi_norm.permute(0, 2, 1, 3)
        resp    = self.head_resp(h).view(B, self.n_max_pisos, 4)
        Ux_p    = resp[:, :, 0] * mask
        Uy_p    = resp[:, :, 1] * mask
        ratio_p = F.softplus(resp[:, :, 2]) * mask
        dy_p    = resp[:, :, 3] * mask
        dx_p    = torch.expm1(ratio_p.clamp(min=0)) * dy_p
        Vb_p    = self.head_Vb(h)
        return (logT_pred,
                Phi_all[:, :, :, 0], Phi_all[:, :, :, 1], Phi_all[:, :, :, 2],
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p)

# ── Verificación de dependencias del main ─────────────────────────────
_deps = ["build_model","compute_loss","get_lambdas_and_active","compute_scalers",
         "make_dataloaders","compute_metrics_ood","ResBlock",
         "CosineAnnealingWarmRestarts","device","SEED","N_INPUTS"]
_missing = [d for d in _deps if d not in dir()]
assert not _missing, f"Faltan definiciones del NB5: {_missing} — corre las celdas del main primero"
print("PINNModal_noT1enc definida. Dependencias del main: OK")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (B) Variantes + curriculum + entrenamiento (Adam+Cosine)
# ══════════════════════════════════════════════════════════════════════
EP_ABL_V6  = 600    # idéntico al main
EP_WARM_V6 = 200    # warm-up data-only para filas 1-5 (curriculum plano)
CKPT_MIN   = 250    # umbral de checkpoint best-val (idéntico al main)

# λ por término = los del main. data_T1=1.0 constante post-warmup en TODAS las filas
# (sin el parámetro muerto "T1"). Cada fila difiere en UN solo término activo.
ABL_V6_VARIANTS = [
    {"id":1, "label":"(1) Supervised NN",      "use_enc_T1":False, "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":False,"ortho":False}},
    {"id":2, "label":"(2) + Encoder T1",       "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":False,"ortho":False}},
    {"id":3, "label":"(3) + L_eig",            "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.1,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":True,"ortho":False}},
    {"id":4, "label":"(4) + L_phi (MAC)",      "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.1,"eig":0.1,"ortho":0.0,"ratio":1.0},
     "act":{"phi":True,"eig":True,"ortho":False}},
    {"id":5, "label":"(5) + L_ortho",          "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.1,"eig":0.1,"ortho":1e-5,"ratio":1.0},
     "act":{"phi":True,"eig":True,"ortho":True}},
    {"id":6, "label":"(6) Full PINN v4",       "use_enc_T1":True,  "staged":True,
     "lam":None, "act":None},
]

def get_lambdas_ablation_v6(epoch, cfg):
    """Fila 6: curriculum v20 del main. Filas 1-5: 2 fases (warm-up data, luego fijo)."""
    if cfg["staged"]:
        return get_lambdas_and_active(epoch)
    if epoch < EP_WARM_V6:
        lam = {"data_resp":2.0,"data_T_rest":1.0,"data_T1":0.0,
               "phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0}
        act = {"phi":False,"eig":False,"ortho":False}
    else:
        lam = dict(cfg["lam"]); lam["data_T1"] = 1.0
        act = dict(cfg["act"])
    return lam, act

def train_ablation_v6(cfg, idx_tr, idx_val, idx_tst, rep):
    """Entrena una variante. Régimen idéntico al main. Devuelve métricas OOD en test."""
    torch.manual_seed(1000*rep + cfg["id"]); np.random.seed(1000*rep + cfg["id"])
    scalers = compute_scalers(idx_tr)                       # scalers SOLO de train
    dl_tr, dl_val, dl_tst = make_dataloaders(idx_tr, idx_val, idx_tst, scalers)

    if cfg["use_enc_T1"]:
        model = build_model().to(device)                   # PINNModal_v4b
    else:
        model = PINNModal_noT1enc(n_inputs=N_INPUTS).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sch = CosineAnnealingWarmRestarts(opt, T_0=200, T_mult=2, eta_min=1e-6)

    has_encT1 = hasattr(model, "encoder_T1")
    if has_encT1:
        pT1 = list(model.encoder_T1.parameters()) + list(model.head_T1.parameters())

    best_val, best_sd = float("inf"), None
    for ep in range(EP_ABL_V6):
        lam, act = get_lambdas_ablation_v6(ep, cfg)
        if has_encT1:
            # entrenamiento en dos velocidades: encoder_T1 activo desde ep>=150 (staged)
            # o desde el warm-up (no-staged), igual que el espíritu del main
            unfreeze = (ep >= 150) if cfg["staged"] else (ep >= EP_WARM_V6)
            for p in pT1: p.requires_grad = unfreeze
        model.train()
        for batch in dl_tr:
            opt.zero_grad()
            out = model(batch["X"].to(device), batch["mask"].to(device))
            loss, _ = compute_loss(batch, out, lam, scalers, act)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()
        # checkpoint best-val (loss ponderada en val), idéntico al main
        model.eval(); vt, nb_ = 0.0, 0
        with torch.no_grad():
            for batch in dl_val:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                l, _ = compute_loss(batch, out, lam, scalers, act)
                vt += l.item(); nb_ += 1
        if ep >= CKPT_MIN and vt/max(nb_,1) < best_val:
            best_val = vt/max(nb_,1)
            best_sd  = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_sd is not None:
        model.load_state_dict(best_sd)
    return compute_metrics_ood(model, dl_tst, scalers)

print("Variantes y funciones de ablación V6 listas.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (C) Ejecución: 6 filas × 4 reps en Trial 16
# ══════════════════════════════════════════════════════════════════════
RUN_ABL_V6  = False    # <- True para ejecutar (~1.5 h)
ABL_TID     = 16       # Trial 16: zona_1_3_vs_zona_2 (mismo del ablation previo)
ABL_V6_PATH = TRIALS_DIR / vfile("ablation_v6", "json")
N_REP_ABL   = 4

if RUN_ABL_V6:
    _tdef = next(t for t in TRIALS_DEF if t["id"] == ABL_TID)
    idx_A = np.where(np.atleast_1d(_tdef["mask_A"]))[0]
    idx_B = np.where(np.atleast_1d(_tdef["mask_B"]))[0]

    # Splits reproducibles: MISMO generador que el loop principal
    rng = np.random.default_rng(SEED + ABL_TID * 100)
    A_sh = rng.permutation(idx_A)
    B_sh = rng.permutation(idx_B)
    A1 = A_sh[:N_TRAIN]; A2 = A_sh[N_TRAIN:2*N_TRAIN]
    n_blk = N_VAL_OOD + N_TEST_OOD
    # 4 reps: alterna dominios de train A1/A2 (factorial) y 4 bloques B disjuntos
    rep_splits = []
    for rep in range(N_REP_ABL):
        tr  = A1 if rep % 2 == 0 else A2
        blk = B_sh[rep*n_blk:(rep+1)*n_blk]
        val = blk[:N_VAL_OOD]; tst = blk[N_VAL_OOD:N_VAL_OOD+N_TEST_OOD]
        # ── Anti-fuga: intersección vacía verificada por código ──
        assert np.intersect1d(tr,  val).size == 0, f"fuga train/val rep{rep}"
        assert np.intersect1d(tr,  tst).size == 0, f"fuga train/test rep{rep}"
        assert np.intersect1d(val, tst).size == 0, f"fuga val/test rep{rep}"
        rep_splits.append((tr, val, tst))
    print(f"Trial {ABL_TID}: Pool A={len(idx_A)}  Pool B={len(idx_B)}  "
          f"| {N_REP_ABL} reps, anti-fuga OK\n")

    abl_results = {}
    for cfg in ABL_V6_VARIANTS:
        reps_m = {"MAC_avg":[], "T1_MAPE":[], "Vb_MAPE":[], "drift_MAPE":[]}
        for rep, (tr, val, tst) in enumerate(rep_splits):
            m = train_ablation_v6(cfg, tr, val, tst, rep)
            for k in reps_m: reps_m[k].append(float(m[k]))
        abl_results[str(cfg["id"])] = {
            "label": cfg["label"],
            "media": {k: float(np.mean(v)) for k, v in reps_m.items()},
            "std":   {k: float(np.std(v))  for k, v in reps_m.items()},
            "reps":  reps_m,
        }
        r = abl_results[str(cfg["id"])]
        print(f"{cfg['label']:22s} | MAC={r['media']['MAC_avg']:.3f}  "
              f"T1={r['media']['T1_MAPE']:.1f}%  Vb={r['media']['Vb_MAPE']:.1f}%  "
              f"drift={r['media']['drift_MAPE']:.1f}%")

    with open(ABL_V6_PATH, "w") as f:
        json.dump(abl_results, f, indent=2)
    print(f"\nGuardado: {ABL_V6_PATH.name}")
else:
    print("RUN_ABL_V6 = False  (no ejecutado)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (D) Candado de reproducción + tabla + LaTeX
# ══════════════════════════════════════════════════════════════════════
if ABL_V6_PATH.exists():
    with open(ABL_V6_PATH) as f:
        abl = json.load(f)

    # ── Candado: fila 6 vs Tabla 8 (Trial 16 del loop principal) ──────
    with open(TRIALS_DIR / vfile("all_results", "json")) as f:
        _ar = json.load(f)
    _main16 = next(v for v in _ar.values() if v.get("trial_id") == ABL_TID)
    f6 = abl["6"]["media"]
    print("=== CANDADO DE REPRODUCCIÓN (fila 6 vs Tabla 8, Trial 16) ===")
    _pass = True
    for k in ["MAC_avg","T1_MAPE","Vb_MAPE","drift_MAPE"]:
        mu  = _main16["media"][k]; sd = _main16["std"][k]
        val = f6[k]; ok = abs(val - mu) <= max(sd, 1e-9)
        _pass = _pass and ok
        print(f"  {k:12s}: ablación={val:8.3f}  main={mu:7.3f}±{sd:.3f}  "
              f"{'OK' if ok else 'FUERA DE ±1sigma'}")
    print(f"\n  Candado: {'PASA -> tabla interpretable' if _pass else 'NO PASA -> revisar pipeline'}\n")

    # ── Tabla agregada ────────────────────────────────────────────────
    rows = []
    for vid in ["1","2","3","4","5","6"]:
        r = abl[vid]
        rows.append({
            "Variante":  r["label"],
            "MAC_avg":   f"{r['media']['MAC_avg']:.3f}±{r['std']['MAC_avg']:.3f}",
            "T1_MAPE":   f"{r['media']['T1_MAPE']:.1f}±{r['std']['T1_MAPE']:.1f}",
            "Vb_MAPE":   f"{r['media']['Vb_MAPE']:.1f}±{r['std']['Vb_MAPE']:.1f}",
            "drift_MAPE":f"{r['media']['drift_MAPE']:.1f}±{r['std']['drift_MAPE']:.1f}",
        })
    df_abl = pd.DataFrame(rows)
    print(df_abl.to_string(index=False))

    csv_path = TRIALS_DIR / vfile("ablation_v6", "csv")
    df_abl.to_csv(csv_path, index=False)
    print(f"\nCSV: {csv_path.name}")

    # ── LaTeX (compatible con Tabla 8) ────────────────────────────────
    print("\n% ---- LaTeX ----")
    print("\\begin{tabular}{lcccc}")
    print("\\hline")
    print("Variante & MAC & $T_1$ MAPE (\\%) & $V_b$ MAPE (\\%) & drift MAPE (\\%) \\\\")
    print("\\hline")
    for _, row in df_abl.iterrows():
        print(f"{row['Variante']} & {row['MAC_avg']} & {row['T1_MAPE']} & "
              f"{row['Vb_MAPE']} & {row['drift_MAPE']} \\\\")
    print("\\hline")
    print("\\end{tabular}")
else:
    print("Sin resultados. Corre la celda ablv6-run con RUN_ABL_V6=True.")
